# Experiment 8 — QR Encoding-Mask Invariance (Controlled Counterfactual)

**Research question:** With URL, phishing label, QR version, ECC, encoding mode,
and raw-pixel model fixed, does changing only the valid QR mask pattern change
model predictions?

Standalone notebook/script. Does not depend on the old Baseline notebook's
executed variables. No CNN training. The raw-pixel XGBoost is loaded from
Exp6, or retrained once using exactly the Exp6 configuration and development
split. All eight masks (0–7) are tested on each selected URL.

**Controls:** Version 13; ECC-L; no quiet-zone in the 69x69 model input;
`qrcode.QRCode(..., mask_pattern=mask)`; identical URL content;
add_data(url, optimize=20); same frozen raw and lexical models.
Original 69x69 QR must **exactly equal** one of the eight generated masks.
This removes encoding-engine / version / error-correction confounds.

**Interpretation:** Visual counterfactual sensitivity measures robustness to
semantically invariant QR mask selection, not an attack success rate. Alternate
masks may be out of distribution for a model trained on the encoder-selected
originals. Lexical predictions for correctly decoded unchanged URLs must be
identical by construction; QR decoding success is measured separately.

**Run modes:** `pilot` (350 samples FROM DEVELOPMENT, feasibility only),
`full` (all 1,777 matched EXP6 holdout, reportable as exploratory due to
prior holdout inspection in earlier experiments).

Dependencies: numpy pandas qrcode[pil] opencv-python joblib scikit-learn
xgboost matplotlib. Never unpickle data/models from untrusted sources.



## 1. Settings and imports — change only RUN_MODE / optional file paths



In [ ]:
from pathlib import Path
import json, pickle, time, warnings, platform, sys
import numpy as np
import pandas as pd
import cv2
import qrcode
import joblib
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, matthews_corrcoef

RUN_MODE = 'full'        # FIRST RUN: 'pilot'; FINAL RUN: change to 'full'
PILOT_N = 350            # development samples (class-balanced)
SEED = 42                # matches Experiment 6
N_BOOTSTRAPS = 300       # hostname-cluster bootstrap (full mode)
VERIFY_ALL_DECODING = True
RETRAIN_RAW_IF_MISSING = True
SHOW_PROGRESS_EVERY = 50

# Paths can be set explicitly if file auto-discovery fails.
PATH_OVERRIDES = {
    'qr_codes_29.pickle': None,
    'qr_codes_29_labels.pickle': None,
    'qr_full_feature_dataset_v2.csv': None,
    'matched_holdout_predictions.csv': None,
    'trad_style_raw_pixel_xgb_grouped.pkl': None,  # optional if retraining
    'FINAL_10feature_XGB_for_deployment.pkl': None,
    'phase14B_final_robust_features.json': None,
}

ROOTS = [
    Path.cwd(), Path.cwd().parent,
    Path.cwd()/'experiment6_matched_trad_vs_lexical',
    Path.cwd().parent/'experiment6_matched_trad_vs_lexical',
    Path.home()/'Desktop'/'FYP',
    Path.home()/'Desktop'/'FYP'/'QuishingDataset',
    Path.home()/'OneDrive'/'Desktop'/'FYP',
    Path.home()/'OneDrive'/'Desktop'/'FYP'/'QuishingDataset',
    Path.home()/'Documents',Path.home()/'Downloads',
    Path(r'C:\Users\LENOVO\Desktop\FYP'),
    Path(r'C:\Users\LENOVO\Desktop\FYP\QuishingDataset'),
    Path(r'C:\Users\LENOVO\FYP\QuishingDataset'),
]

def locate(name, optional=False):
    explicit = PATH_OVERRIDES[name]
    if explicit:
        p = Path(explicit).expanduser()
        if not p.is_file(): raise FileNotFoundError(f'Incorrect override: {name} => {p}')
        return p.resolve()
    for root in ROOTS:
        for p in (root/name,
                  root/'experiment6_matched_trad_vs_lexical'/name,
                  root/'QuishingDataset'/name):
            if p.is_file(): return p.resolve()
    if optional: return None
    raise FileNotFoundError(f'{name} not found. Edit PATH_OVERRIDES in Section 1. '
                            f'Notebook directory: {Path.cwd().resolve()}')

assert RUN_MODE in ('pilot','full')
OUT = Path.cwd() / f'experiment8_qr_mask_invariance_{RUN_MODE}'
OUT.mkdir(parents=True,exist_ok=True)
OPTIONAL={'trad_style_raw_pixel_xgb_grouped.pkl'}
FILES={name:locate(name,optional=name in OPTIONAL) for name in PATH_OVERRIDES}
for name,p in FILES.items(): print(f'{name}: {p if p else "(will retrain raw XGBoost)"}')
print('MODE:',RUN_MODE,'| OUTPUT:',OUT.resolve())
cv2.setNumThreads(1)



## 2. Load aligned original data and lock Experiment 6 split

The exact 1,777 holdout original_index, label, hostname, URL and XGBoost
probabilities are checked. No new random outer split is introduced.



In [ ]:
with open(FILES['qr_codes_29.pickle'],'rb') as f: QR=np.asarray(pickle.load(f))
with open(FILES['qr_codes_29_labels.pickle'],'rb') as f: QR_Y=np.asarray(pickle.load(f),dtype=int)
D=pd.read_csv(FILES['qr_full_feature_dataset_v2.csv'])
E6=pd.read_csv(FILES['matched_holdout_predictions.csv'])
with open(FILES['phase14B_final_robust_features.json'],encoding='utf-8') as f:
    FEATURES=list(json.load(f)['features'])
assert QR.shape==(9987,69,69),QR.shape
assert QR_Y.shape==(9987,),QR_Y.shape
assert len(D)==9713,len(D)
assert D.original_index.is_unique and D[['original_index','url','hostname','label']+FEATURES].notna().all().all()
assert np.isin(QR,[0,1]).all(), 'Expected original raw inputs as binary 0=black, 1=white'
orig=D.original_index.to_numpy(dtype=int)
labels=D.label.to_numpy(dtype=int)
hosts=D.hostname.astype(str).to_numpy()
assert np.array_equal(QR_Y[orig],labels),'QR/CSV labels misaligned!'
assert len(FEATURES)==10
outer=GroupShuffleSplit(n_splits=1,test_size=.2,random_state=SEED)
DEV,TEST=next(outer.split(D,labels,groups=hosts))
assert (len(DEV),len(TEST))==(7936,1777)
assert set(hosts[DEV]).isdisjoint(set(hosts[TEST]))
assert E6.original_index.is_unique and len(E6)==1777
assert set(E6.original_index.astype(int))==set(orig[TEST]),'Exp6 holdout IDs differ'
E6=E6.set_index('original_index').loc[orig[TEST]].reset_index()
assert np.array_equal(E6.label.to_numpy(dtype=int),labels[TEST])
assert np.array_equal(E6.hostname.astype(str).to_numpy(),hosts[TEST])
assert np.array_equal(E6.url.astype(str).to_numpy(),D.url.iloc[TEST].astype(str).to_numpy())
assert abs(roc_auc_score(labels[TEST],E6.raw_pixel_probability)-.88218)<.0002
assert abs(roc_auc_score(labels[TEST],E6.lexical_probability)-.95095)<.0002
print(f'PASS: Exp6 exact holdout IDs and label/URL/hostname alignment, N={len(TEST)}')



## 3. Load the frozen reference models and select evaluation rows

Pilot draws a balanced development subset for methodology checks, NOT a
generalisation estimate. Full mode evaluates the locked 1,777 holdout.



In [ ]:
if FILES['trad_style_raw_pixel_xgb_grouped.pkl'] is not None:
    raw_model=joblib.load(FILES['trad_style_raw_pixel_xgb_grouped.pkl'])
    raw_source='loaded original Exp6 raw model'
else:
    if not RETRAIN_RAW_IF_MISSING:
        raise FileNotFoundError('Exp6 raw model not found; set RETRAIN_RAW_IF_MISSING=True')
    print('Training Exp6-equivalent raw XGBoost on development data only...')
    raw_model=XGBClassifier(learning_rate=.2,n_estimators=150,
                            eval_metric='logloss',random_state=SEED,n_jobs=4)
    raw_model.fit(QR[orig[DEV]].reshape(len(DEV),4761).astype(np.float32),labels[DEV])
    joblib.dump(raw_model,OUT/'exp6_equivalent_raw_model.pkl')
    raw_source='retrained once with Exp6 architecture / development split'

raw_test_probs=raw_model.predict_proba(QR[orig[TEST]].reshape(len(TEST),4761).astype(np.float32))[:,1]
raw_auc=roc_auc_score(labels[TEST],raw_test_probs)
print(f'Raw XGBoost holdout AUC: {raw_auc:.6f}')
assert abs(raw_auc-.88218)<.0004, 'STOP: Raw model does not reproduce Experiment 6 clean AUC'
ref_abs_err=np.abs(raw_test_probs-E6.raw_pixel_probability.to_numpy(float))
print('Original raw scores vs Exp6: mean abs diff',float(ref_abs_err.mean()),
      '| max abs diff',float(ref_abs_err.max()))
# Tiny numeric differences across xgboost versions are possible; flag substantive differences.
assert float(ref_abs_err.mean())<.002, 'STOP: Raw-pixel predictions differ from Exp6, inspect model training'

lex_model=joblib.load(FILES['FINAL_10feature_XGB_for_deployment.pkl'])
assert list(getattr(lex_model,'feature_names_in_',FEATURES))==FEATURES
lex_check=lex_model.predict_proba(D.iloc[TEST][FEATURES])[:,1]
lex_err=np.abs(lex_check-E6.lexical_probability.to_numpy(float))
assert float(lex_err.max())<1e-4,'STOP: Frozen lexical model differs from Exp6'

if RUN_MODE=='pilot':
    rng=np.random.default_rng(20261108)
    pilot_pos=np.concatenate([
        rng.choice(DEV[labels[DEV]==0],size=PILOT_N//2,replace=False),
        rng.choice(DEV[labels[DEV]==1],size=PILOT_N-PILOT_N//2,replace=False)
    ])
    selected=np.sort(pilot_pos)
    print('PILOT: development subset; metrics are in-sample, not generalisation estimates.')
else:
    selected=TEST.copy()
N=len(selected)
sub=D.iloc[selected].reset_index(drop=True)
y=sub.label.to_numpy(dtype=int)
sub_orig=sub.original_index.to_numpy(dtype=int)
sub_host=sub.hostname.astype(str).to_numpy()
sub_url=sub.url.astype(str).to_numpy()
orig_features=QR[sub_orig].reshape(N,4761).astype(np.float32)
orig_probs=raw_model.predict_proba(orig_features)[:,1]
lex_probs=lex_model.predict_proba(sub[FEATURES])[:,1]
if RUN_MODE=='full':
    assert np.allclose(orig_probs,E6.raw_pixel_probability.to_numpy(float),atol=.002), \
        'STOP: full-mode sample order differs from Experiment 6'
print('Selected:',N,'| phishing share:',round(float(y.mean()),4),
      '| unique hostnames:',len(np.unique(sub_host)))



## 4. Controlled generator (Version 13, ECC-L, same URL encoding)

Validate that **one of the eight generated masks EXACTLY recreates** each
original native QR grid. If not, the row is excluded and a reason is saved;
the experiment does not silently compare different QR encoders.

qrcode's 1=black is converted to the baseline's 0=black, 1=white.
Note: The CSV's `qr_version` field denotes a minimum-size property; the
original **raw model inputs** are all 69x69 (Version 13).



In [ ]:
QR_VERSION=13
ECC=qrcode.constants.ERROR_CORRECT_L
MASKS=list(range(8))
SCALE=8
BORDER=4
SIDE=(69+2*BORDER)*SCALE

def make_mask_grid(url,mask):
    qr=qrcode.QRCode(version=QR_VERSION,error_correction=ECC,
                     box_size=1,border=0,mask_pattern=int(mask))
    qr.add_data(str(url),optimize=20)
    qr.make(fit=False)   # Fail on capacity overrun; never alter version
    matrix=np.asarray(qr.get_matrix(),dtype=np.uint8)
    if matrix.shape!=(69,69): raise ValueError(f'Unexpected generated QR shape: {matrix.shape}')
    return (1-matrix).astype(np.uint8)  # baseline: black=0, white=1

def render_for_decoder(grid):
    image=np.pad(np.asarray(grid,dtype=np.uint8)*255,
                 ((BORDER,BORDER),(BORDER,BORDER)),
                 mode='constant',constant_values=255)
    return cv2.resize(image,(SIDE,SIDE),interpolation=cv2.INTER_NEAREST)

# Synthetic self-test: each mask valid; original anchor must be exact for example.
smoke_url=sub_url[0]
smoke_mats=np.stack([make_mask_grid(smoke_url,m) for m in MASKS])
assert smoke_mats.shape==(8,69,69)
assert any(np.array_equal(g,QR[sub_orig[0]]) for g in smoke_mats), \
    'First original QR does not match controlled generator; STOP and inspect parameters.'
print('PASS: QR generator 69x69; original example identical to one valid mask.')



## 5. Generate eight masks, verify original alignment and decoding

Decode is evaluated on a 616x616, white-bordered version of each QR. It is
**never** used to train the raw model or change the encoded URL. Failed decode
is recorded as a lexical-pipeline abstention, not removed from visual metrics.
Full mode can take several minutes because OpenCV decodes 14,216 images.



In [ ]:
start=time.perf_counter()
detector=cv2.QRCodeDetector()
records=[]
anchor_rows=[]
valid_rows=[]
score_batches=[]
for i in range(N):
    url=sub_url[i]
    try:
        matrices=np.stack([make_mask_grid(url,m) for m in MASKS])
    except Exception as ex:
        anchor_rows.append({'sample_row':i,'original_index':int(sub_orig[i]),
                            'valid':False,'reason':type(ex).__name__+': '+str(ex)[:160],
                            'original_mask':np.nan})
        continue
    matches=[m for m in MASKS if np.array_equal(matrices[m],QR[sub_orig[i]])]
    if len(matches)!=1:
        anchor_rows.append({'sample_row':i,'original_index':int(sub_orig[i]),
                            'valid':False,'reason':f'Expected 1 exact original-mask match, got {matches}',
                            'original_mask':np.nan})
        continue
    native_mask=matches[0]
    anchor_rows.append({'sample_row':i,'original_index':int(sub_orig[i]),
                        'valid':True,'reason':'exact original matrix verified',
                        'original_mask':native_mask})
    valid_rows.append(i)
    score_batches.append(matrices.reshape(8,4761).astype(np.float32))
    for mask in MASKS:
        if VERIFY_ALL_DECODING:
            decoded,_,_=detector.detectAndDecode(render_for_decoder(matrices[mask]))
            decoded_ok=(decoded==url)
            decoded_nonempty=bool(decoded)
        else:
            decoded_ok=False
            decoded_nonempty=False
        records.append(dict(sample_row=i,original_index=int(sub_orig[i]),
                            hostname=sub_host[i],label=int(y[i]),
                            mask=mask,original_mask=native_mask,
                            is_original_mask=(mask==native_mask),
                            decoded_correct=bool(decoded_ok),
                            decoded_nonempty=decoded_nonempty,
                            mask_pixel_change=float(np.mean(matrices[mask]!=QR[sub_orig[i]]))))
    if (i+1)%SHOW_PROGRESS_EVERY==0 or i+1==N:
        print(f'Processed {i+1}/{N}; valid={len(valid_rows)}; elapsed={time.perf_counter()-start:.1f}s',flush=True)

anchor_df=pd.DataFrame(anchor_rows)
anchor_df.to_csv(OUT/'encoder_alignment_audit.csv',index=False)
assert len(valid_rows)>0,'No verified original QR anchors; cannot perform controlled experiment.'
coverage=len(valid_rows)/N
print(f'Original QR exact-mask matching coverage: {len(valid_rows)}/{N} = {coverage:.2%}')
assert coverage>=.95, 'STOP: <95% exact original anchors; examine generator compatibility'
rows=pd.DataFrame(records).sort_values(['sample_row','mask']).reset_index(drop=True)
assert len(rows)==8*len(valid_rows)
assert rows.groupby('sample_row')['mask'].nunique().eq(8).all()
assert np.all(rows.loc[rows.is_original_mask,'mask_pixel_change']==0)

# Predict after generation, in moderate batches (avoid 14k x 4761 dense array).
# The counterfactual model is identical for all mask conditions.
score_blocks=[]
for j in range(0,len(score_batches),100):
    chunk=np.concatenate(score_batches[j:j+100],axis=0)
    score_blocks.append(raw_model.predict_proba(chunk)[:,1])
score_vector=np.concatenate(score_blocks)
assert len(score_vector)==len(rows)
rows['raw_probability']=score_vector
rows['raw_prediction']=(score_vector>=.5).astype(int)
rows['raw_correct']=rows.raw_prediction.eq(rows.label)
rows['lexical_probability_given_correct_decode']=np.where(
    rows.decoded_correct.to_numpy(bool),
    np.repeat(lex_probs[np.array(valid_rows,dtype=int)],8),np.nan)
rows['lexical_strict_correct']=(rows.decoded_correct &
    (np.repeat((lex_probs[np.array(valid_rows,dtype=int)]>=.5).astype(int),8)==rows.label))
rows.to_csv(OUT/'sample_level_all_masks.csv',index=False)
print('All eight masks scored, sample-level results saved.')



## 6. Main results: original match, mask-specific performance, per-URL flips

Per-URL flip: at least two masks predict different binary classes for the
**same URL and label**. Score range measures continuous prediction sensitivity.
Mask-specific AUCs use the **same set of eligible URLs** and the same model.
Decoder failure is not conflated with visual model performance.



In [ ]:
valid=np.array(valid_rows,dtype=int)
v_y=y[valid]
v_host=sub_host[valid]
v_orig_probs=orig_probs[valid]
orig_generated=rows.loc[rows.is_original_mask].sort_values('sample_row')
assert np.allclose(orig_generated.raw_probability.to_numpy(),v_orig_probs,atol=1e-6), \
    'STOP: Generated original mask does not reproduce raw model inputs/scores.'

per_url=[]
for sample_row, group in rows.groupby('sample_row',sort=True):
    p=group.raw_probability.to_numpy()
    pred=group.raw_prediction.to_numpy()
    oi=group.loc[group.is_original_mask].iloc[0]
    per_url.append(dict(sample_row=int(sample_row),
                        original_index=int(oi.original_index),
                        hostname=oi.hostname,label=int(oi.label),
                        original_mask=int(oi.original_mask),
                        original_raw_probability=float(oi.raw_probability),
                        original_raw_prediction=int(oi.raw_prediction),
                        any_flip_across_masks=bool(pred.min()!=pred.max()),
                        flip_vs_original_fraction=float(np.mean(pred!=oi.raw_prediction)),
                        any_flip_vs_original=bool((pred!=oi.raw_prediction).any()),
                        prediction_probability_range=float(p.max()-p.min()),
                        prediction_probability_sd=float(np.std(p)),
                        decoder_correct_fraction=float(group.decoded_correct.mean()),
                        any_decoder_failure=bool((~group.decoded_correct).any())))
per_url=pd.DataFrame(per_url)
per_url.to_csv(OUT/'per_url_mask_sensitivity.csv',index=False)

condition_rows=[]
for mask,group in rows.groupby('mask',sort=True):
    decoded_subset=group.loc[group.decoded_correct] if VERIFY_ALL_DECODING else group.iloc[:0]
    lexical_conditional_auc=(
        float(roc_auc_score(decoded_subset.label,decoded_subset.lexical_probability_given_correct_decode))
        if (len(decoded_subset)>0 and decoded_subset.label.nunique()==2) else np.nan)
    condition_rows.append(dict(
        condition=f'Mask {mask}',mask=int(mask),n=len(group),
        raw_auc=float(roc_auc_score(group.label,group.raw_probability)),
        raw_accuracy=float(group.raw_correct.mean()),
        raw_f1=float(f1_score(group.label,group.raw_prediction,zero_division=0)),
        raw_mcc=float(matthews_corrcoef(group.label,group.raw_prediction)),
        raw_predicted_phishing_rate=float(group.raw_prediction.mean()),
        raw_flip_vs_original_rate=float(np.mean(group.raw_prediction.to_numpy()!=
                                     (v_orig_probs>=.5).astype(int))),
        correct_decode_rate=float(group.decoded_correct.mean()) if VERIFY_ALL_DECODING else np.nan,
        lexical_conditional_auc=lexical_conditional_auc,
        lexical_strict_correct_rate=float(group.lexical_strict_correct.mean())
                              if VERIFY_ALL_DECODING else np.nan,
        pixel_change_mean=float(group.mask_pixel_change.mean())))
summary=pd.DataFrame(condition_rows)
summary.to_csv(OUT/'mask_specific_summary.csv',index=False)

original_summary=dict(condition='Original encoder-selected mask',mask=-1,n=len(valid),
                      raw_auc=float(roc_auc_score(v_y,v_orig_probs)),
                      raw_accuracy=float(np.mean((v_orig_probs>=.5)==v_y)),
                      raw_f1=float(f1_score(v_y,(v_orig_probs>=.5).astype(int),zero_division=0)),
                      raw_mcc=float(matthews_corrcoef(v_y,(v_orig_probs>=.5).astype(int))),
                      raw_predicted_phishing_rate=float(np.mean(v_orig_probs>=.5)),
                      raw_flip_vs_original_rate=0.0,
                      correct_decode_rate=float(orig_generated.decoded_correct.mean()) if VERIFY_ALL_DECODING else np.nan,
                      lexical_conditional_auc=(float(roc_auc_score(orig_generated.loc[orig_generated.decoded_correct,'label'],
                         orig_generated.loc[orig_generated.decoded_correct,'lexical_probability_given_correct_decode']))
                         if VERIFY_ALL_DECODING and orig_generated.loc[orig_generated.decoded_correct,'label'].nunique()==2 else np.nan),
                      lexical_strict_correct_rate=float(orig_generated.lexical_strict_correct.mean()) if VERIFY_ALL_DECODING else np.nan,
                      pixel_change_mean=0.0)
summary_all=pd.concat([pd.DataFrame([original_summary]),summary],ignore_index=True)
summary_all.to_csv(OUT/'original_and_mask_summary.csv',index=False)

print('\n=== ENCODER MASK INVARIANCE: MAIN RESULTS ===')
print(summary_all[['condition','n','raw_auc','raw_f1','raw_mcc',
                   'raw_flip_vs_original_rate','correct_decode_rate']]
      .round(4).to_string(index=False))
print('\n=== PER-URL PREDICTION STABILITY ===')
for key in ['any_flip_across_masks','flip_vs_original_fraction',
            'prediction_probability_range','any_decoder_failure']:
    print(f'{key}: {per_url[key].mean():.4f}')
print('Original selected-mask counts:',per_url.original_mask.value_counts().sort_index().to_dict())



## 7. Paired, hostname-cluster bootstrap (FULL mode only)

Report 95% intervals for model changes with fixed samples grouped by URL
hostname. These intervals are exploratory and do not account for the fact
that previous CNN debugging inspected the same holdout. Eight masks are
repeated measures for each URL; never treat 8*N rows as independent samples.



In [ ]:
if RUN_MODE=='full':
    groups,inv=np.unique(v_host,return_inverse=True)
    group_rows=[np.flatnonzero(inv==k) for k in range(len(groups))]
    # 2D arrays: [URL row, mask 0...7]
    prob_matrix=rows.pivot(index='sample_row',columns='mask',values='raw_probability').loc[valid,MASKS].to_numpy()
    pred_matrix=(prob_matrix>=.5)
    rng=np.random.default_rng(20261108)
    original_auc=float(roc_auc_score(v_y,v_orig_probs))
    boot_flips=[]
    boot_auc_delta={m:[] for m in MASKS}
    for b in range(N_BOOTSTRAPS):
        idx=np.concatenate([group_rows[k] for k in rng.integers(0,len(groups),size=len(groups))])
        boot_flips.append(float(np.mean(per_url.any_flip_across_masks.to_numpy()[idx])))
        yy=v_y[idx]
        if len(np.unique(yy))!=2: continue
        base_auc=roc_auc_score(yy,v_orig_probs[idx])
        for m in MASKS:
            boot_auc_delta[m].append(float(roc_auc_score(yy,prob_matrix[idx,m])-base_auc))
    ci_rows=[dict(metric='URL_any_classification_flip_rate',condition='all_masks',
                  estimate=float(per_url.any_flip_across_masks.mean()),
                  ci_low=float(np.quantile(boot_flips,.025)),
                  ci_high=float(np.quantile(boot_flips,.975)),
                  bootstrap_unit='hostname')]
    for m in MASKS:
        vals=boot_auc_delta[m]
        assert len(vals)>=N_BOOTSTRAPS*.8
        ci_rows.append(dict(metric='AUC_delta_vs_original',condition=f'mask_{m}',
                            estimate=float(summary.loc[summary['mask']==m,'raw_auc'].iloc[0]-original_auc),
                            ci_low=float(np.quantile(vals,.025)),
                            ci_high=float(np.quantile(vals,.975)),
                            bootstrap_unit='hostname'))
    ci=pd.DataFrame(ci_rows)
    ci.to_csv(OUT/'hostname_bootstrap_ci.csv',index=False)
    print('\n=== HOSTNAME BOOTSTRAP CIs ===')
    print(ci.round(4).to_string(index=False))
else:
    print('PILOT MODE: inferential bootstrap skipped (in-sample feasibility test).')



## 8. Plots and saved reproducibility metadata

These charts show representation sensitivity, not a proof that URL lexical
features universally outperform vision. If all masks are decoded correctly,
the lexical classifier necessarily returns identical features and predictions.



In [ ]:
fig,ax=plt.subplots(figsize=(9,4.5))
ax.bar(summary.condition,summary.raw_auc)
ax.axhline(original_summary['raw_auc'],ls='--',label='Original selected mask AUC')
ax.set_ylabel('Raw-pixel ROC–AUC')
ax.set_ylim(0,1)
ax.set_title(f'QR mask counterfactuals — {RUN_MODE}, {len(valid)} eligible URLs')
ax.legend()
plt.xticks(rotation=30)
fig.tight_layout()
fig.savefig(OUT/'mask_auc.png',dpi=180,bbox_inches='tight')
plt.show()

fig,ax=plt.subplots(figsize=(9,4.5))
ax.bar(summary.condition,summary.raw_flip_vs_original_rate)
ax.set_ylabel('Raw-pixel prediction flip rate vs original mask')
ax.set_ylim(0,max(.05,min(1,summary.raw_flip_vs_original_rate.max()*1.2)))
ax.set_title(f'Mask-induced label changes — {RUN_MODE}, per-mask paired URLs')
plt.xticks(rotation=30)
fig.tight_layout()
fig.savefig(OUT/'mask_prediction_flip_rates.png',dpi=180,bbox_inches='tight')
plt.show()

metadata={
    'study':'Experiment 8 controlled QR mask invariance',
    'mode':RUN_MODE,'n_selected':N,'n_matched_original':len(valid),
    'original_anchor_coverage':float(coverage),
    'original_qr_shape':[69,69],'qrcode_version':QR_VERSION,
    'ecc':'L','encoding':'qrcode.QRCode.add_data(url,optimize=20)',
    'qr_masks':MASKS,'raw_pixel_convention':'0 black, 1 white',
    'lexical_pipeline':'same frozen 10-feature model; decode correctness reported separately',
    'raw_model_source':raw_source,
    'raw_clean_auc_full_holdout':float(raw_auc),
    'experiment6_reference_auc':.88218,
    'decode_all':VERIFY_ALL_DECODING,
    'decoder_render_side_pixels':SIDE,
    'bootstrap_replicates':N_BOOTSTRAPS if RUN_MODE=='full' else 0,
    'bootstrap_unit':'hostname',
    'dataset_label_definition':'0 benign, 1 phishing',
    'note':'Pilot uses development training samples; full holdout previously inspected for CNN diagnostics; exploratory not pristine final test.',
    'packages':{'python':sys.version.split()[0],'qrcode':str(getattr(qrcode,'__version__','not exposed')),
                'opencv':cv2.__version__},
    'elapsed_seconds':round(time.perf_counter()-start,2),
}
(OUT/'experiment_protocol.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf-8')
print('\nDONE. Files saved to:',OUT.resolve())
print('Key files: per_url_mask_sensitivity.csv, original_and_mask_summary.csv,')
print('sample_level_all_masks.csv, encoder_alignment_audit.csv, experiment_protocol.json')
print('Important: QR masks are controlled valid encoding variants; counterfactual masks may be OOD relative to original selected masks.')
